# 数据清洗与收益率

读取下载的股票面板与指数行情，生成 `data/processed/panel.parquet` 和 `index.parquet`，供后续收益标签、因子与回测共用。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

START = "20220101"
END = "20251231"

DAILY = PROJECT_ROOT / "data/raw/日频行情_2022_2025.parquet"
INDEX = PROJECT_ROOT / "data/raw/上证A股指数_2022_2025.parquet"

PANEL = PROJECT_ROOT / "data/processed/panel.parquet"
INDEX_ = PROJECT_ROOT / "data/processed/index.parquet"

## 读取原始数据

股票面板和指数行情分别读取，以下先处理股票数据，再处理指数数据。


In [ ]:
panel = pd.read_parquet(DAILY)
index = pd.read_parquet(INDEX)

## 排序与异常值整理

将无穷值替换为缺失，按交易日和股票代码排序，保证每只股票的记录在时间上递增。


In [ ]:
panel = panel.replace(
    [np.inf, -np.inf], np.nan
)

panel = panel.sort_values(["trade_date", "ts_code"]).reset_index(drop=True)

## 标记停牌与 ST

根据原始停牌记录和特殊处理类型生成布尔标记；`ohlc` 是辅助字段，用于识别开、高、低、收是否均有效。


In [ ]:
panel["ohlc"] = panel[["open", "high", "low", "close"]].notna().all(axis=1)

panel["suspend"] = panel["suspend_type"].notna()

panel['ST'] = panel["ST_type"].notna()

## 调整状态与填充交易量字段

有日内停牌时间且开高低收完整的记录按非停牌处理。`vol`、`amount`、`turnover_rate` 的缺失统一填为 0，价格列保持原值。


In [ ]:
panel.loc[panel['suspend_timing'].notna() &  (panel['ohlc']== True), 'suspend'] = False

zero_fill_cols = ['vol', 'amount', 'turnover_rate']
panel[zero_fill_cols] = panel[zero_fill_cols].fillna(0.0)

## 生成后复权价格

开、高、低、收和日均价分别乘 `adj_factor`，形成对应的 `_hfq` 列。


In [ ]:
hfq_cols = ['open', 'high', 'low', 'close', 'vwap']

for col in hfq_cols:
    panel[f'{col}_hfq'] = panel[col] * panel['adj_factor']

## 计算股票收益率

在各股票内部前向填充收盘价后，计算相邻市场交易日收益率：

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t}),\qquad
r_{i,t}=\widetilde C_{i,t}/\widetilde C_{i,t-1}-1.
$$

该填充只用于 `ret`，不会覆盖价格列。停牌期间通常为 0，复牌日相对最后有效收盘价计算；首次有效价格之前仍缺失。


In [ ]:
panel["ret"] = (
    panel.groupby("ts_code", sort=False)["close_hfq"]
    .transform(lambda s: s.ffill().pct_change(fill_method=None))
)

## 整理并保存股票面板

删除状态识别和复权使用的辅助字段，预览后将结果写入 `PANEL`。后续股票表均以 `trade_date + ts_code` 对齐。


In [ ]:
panel = panel.drop(columns = ["ohlc", 'suspend_timing', "suspend_type", "ST_type", 'adj_factor'])

In [ ]:
panel.head()

In [ ]:
PANEL.parent.mkdir(parents=True, exist_ok=True)
panel.to_parquet(PANEL,index=False)

## 处理指数收益率

整理异常值并按日期排序，再用相邻收盘点位计算 `ret`。指数价格不填充，首日收益率缺失。


In [ ]:
index = index.replace(
    [np.inf, -np.inf], np.nan
)

In [ ]:
index = index.sort_values(["trade_date"]).reset_index(drop=True)

In [ ]:
index["ret"] = index["close"].pct_change(fill_method=None)

In [ ]:
index.head()

## 保存指数行情

保存至 `INDEX_`，供后续因子计算及同期指数对照使用。


In [ ]:
index.to_parquet(INDEX_,index=False)